# Example 8.1 / 1: Select a Solver and Check What It Did

Companion notebook for **8.1 Choosing a Solver** in the AMPL lab Documentation, section "Select a Solver and Check What It Did". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Select a Solver and Check What It Did

`ampl.solve(solver="highs")` selects a solver for that call; in pure AMPL,
`option solver highs;` selects it for every later `solve`. Use the exact driver name installed
in the AMPL environment: a product name and its driver name are not always interchangeable.

A `solved` status says the solver finished, not that the solver suits the model. This version of
the steel model has a nonlinear objective: revenue grows more slowly as a product approaches its
market size. It is solved by HiGHS, an LP/MIP solver, and by Ipopt, a nonlinear solver. In
`data.py` and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`;
`verbose=False` hides the solver's log, and `ampl.solve_result` gives the status:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
# profit with saturating sales: each extra ton earns less near the market size
maximize Total_Profit:
    sum {p in PROD} profit[p] * market[p] * (1 - exp(-Make[p] / market[p]));
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
for solver in ["highs", "ipopt"]:
    ampl.solve(solver=solver, verbose=False)
    profit = ampl.obj["Total_Profit"].value()
    print(f"{solver}: {ampl.solve_result}, profit {profit:.1f}")
    print(ampl.get_value("solve_message").strip())

highs: solved, profit 145127.5
HiGHS 1.14.0: optimal solution; objective 144910.7485
7 simplex iterations
0 barrier iterations
 
------------ WARNINGS ------------
WARNING.  3 case(s) of "PLApprox_Exp". One of them:
  An expression of type 'Exp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.010000).
ipopt: solved, profit 146171.0
Ipopt 3.12.13: Optimal Solution Found


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver_msg 0;
option solver highs;
solve;
display solve_result, Total_Profit;
option solver ipopt;
option ipopt_options 'print_level=0';
solve;
display solve_result, Total_Profit;

HiGHS 1.14.0: 
------------ WARNINGS ------------
WARNING.  3 case(s) of "PLApprox_Exp". One of them:
  An expression of type 'Exp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.010000).
solve_result = solved
Total_Profit = 145127

Ipopt 3.12.13: print_level=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************


suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
solve_result = solved
Total_Profit = 146171

